# 0.11 · 信息论 / Information Theory

> **课程定位 / Where this fits**
> 第 11 课，**Part 0 · 基础准备**。
> Lesson 11, **Part 0 · Foundations**.
>
> 概率（0.9）告诉你"事件发生的可能性"，**信息论告诉你"事件传达了多少信息"**。机器学习里的几乎所有"距离/损失"都是某种信息论量：
> Probability tells you the chance of events; **information theory tells you how much information they carry**. Most ML "distances/losses" are info-theoretic:
> - 分类损失 / cross-entropy → **就是信息论**
> - 决策树分裂 / info gain → **就是信息论**
> - VAE 的 ELBO → KL 散度
> - GAN / 知识蒸馏 / RLHF → KL 散度
> - LLM 评估 / perplexity → 熵

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $H(X)$ —— 熵 / entropy（默认 $\log_2$，单位 bit；用 $\ln$ 单位是 nat）
> - $H(X, Y)$ —— 联合熵 / joint entropy
> - $H(X \mid Y)$ —— 条件熵 / conditional entropy
> - $I(X; Y)$ —— 互信息 / mutual information
> - $D_{\mathrm{KL}}(p \,\|\, q)$ —— KL 散度 / relative entropy
> - $H(p, q)$ —— 交叉熵 / cross-entropy（注意：和**联合熵** $H(X,Y)$ 不同！）

> 💡 **面试相关 / Interview-relevant**
> - "为什么分类用交叉熵不用 MSE" ★★★★★
> - "KL 散度为什么不对称" ★★★★
> - "熵的直觉解释" ★★★
> - "互信息和相关系数的区别" ★★★
>
> Top hits: cross-entropy-over-MSE, asymmetry of KL, entropy intuition, MI vs correlation.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 用一句话解释**熵 = 不确定性 / 编码长度下界**。
   Explain entropy in one sentence as uncertainty / minimum code length.
2. 推导**熵 / 条件熵 / 互信息**三者的关系（含 $I = H(X) - H(X\mid Y)$）。
   Derive the relationships among entropy, conditional entropy, and mutual information.
3. 解释 **KL 散度的非对称性**，给出 forward-KL vs reverse-KL 的几何直觉。
   Explain the asymmetry of KL; give the geometric intuition of forward vs reverse KL.
4. 推出 **交叉熵 = 熵 + KL**，明白为什么分类用 cross-entropy。
   Show cross-entropy = entropy + KL and explain why classification uses CE.
5. 用 **信息增益**在 Iris 上构建一个 1 层决策树并解释结果。
   Build a one-level decision tree on Iris using information gain.
6. 计算 **LLM perplexity** 并理解它是"困惑度 = 2^entropy"。
   Compute LLM perplexity and connect it to 2^entropy.

---

## 目录 / Table of Contents

1. [自信息 / Self-information](#1)
2. [熵 ⭐ / Shannon Entropy](#2)
3. [联合熵 & 条件熵 / Joint & Conditional](#3)
4. [互信息 ⭐ / Mutual Information](#4)
5. [KL 散度 ⭐ / Kullback–Leibler Divergence](#5)
6. [**交叉熵 = 熵 + KL** ⭐](#6)
7. [为什么分类用交叉熵不用 MSE / Why CE not MSE](#7)
8. [Jensen 不等式 & Gibbs 不等式](#8)
9. [连续随机变量的熵 / Differential Entropy](#9)
10. [信息论在 ML 里的应用 / ML Applications](#10)
11. [实战：信息增益构建决策树（Iris）/ Hands-on](#11)
12. [小结 / Summary](#12)


<a id="1"></a>
## 1. 自信息 / Self-information

直觉：**越罕见的事件，发生时传达的信息越多**。
Intuition: **rarer events carry more information**.

例：朋友告诉你"今天太阳从东边升起" → 0 信息。"今天彩虹同时出现两道" → 大新闻。
"The sun rose in the east" — zero info. "There were two rainbows" — big news.

**Shannon 自信息** / Shannon self-information:
$$
\boxed{\;I(x) \;=\; -\log_2 \Pr(X = x)\; \text{ bit}\;}
$$

性质 / Properties:
- $\Pr(x) = 1 \Rightarrow I(x) = 0$ —— 确定事件没信息
- $\Pr(x) \to 0 \Rightarrow I(x) \to \infty$ —— 极罕见 = 极大信息
- 对**独立**事件 $A, B$：$I(A \cap B) = I(A) + I(B)$（来自 log）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

# 几个事件的自信息 / Self-info of a few events
events = {
    "fair coin → heads":    0.5,
    "fair die → 6":         1/6,
    "roll 7 with 2 dice":   6/36,
    "lottery 1 / 14M":      1/14_000_000,
    "the sun rises today":  1 - 1e-12,
}
print(f"{'event':<28} {'P':>16} {'I(x) [bits]':>14}")
print("-" * 60)
for name, p in events.items():
    print(f"{name:<28} {p:>16.2e} {-np.log2(p):>14.4f}")


<a id="2"></a>
## 2. 熵 ⭐ / Shannon Entropy

**熵 = 自信息的期望** = 一次随机实验**平均**带来多少信息。
**Entropy = expected self-information** = average info per draw.

$$
\boxed{\;H(X) \;=\; \mathbb{E}_{p(x)}\bigl[-\log_2 p(x)\bigr] \;=\; -\sum_x p(x)\log_2 p(x) \;\text{ bit}\;}
$$

（连续情况积分代和；ML 里常用 $\ln$，单位 nat。）

### 三种等价直觉 / Three equivalent intuitions

1. **不确定性度量** / measure of uncertainty: 越随机熵越大
2. **平均编码长度下界** / lower bound on average code length: Shannon 编码定理
3. **"surprise"** / expected surprise

### 二元情况 / Binary

$X \in \{0, 1\}$ with $\Pr(X = 1) = p$：
$$H_b(p) = -p \log p - (1-p)\log(1-p)$$


In [ ]:
# 画二元熵函数 / Plot binary entropy
ps = np.linspace(0.001, 0.999, 200)
H = -ps*np.log2(ps) - (1-ps)*np.log2(1-ps)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, H, "b-", linewidth=2)
ax.axhline(1.0, color="gray", lw=0.5, linestyle="--")
ax.axvline(0.5, color="gray", lw=0.5, linestyle="--")
ax.scatter([0.5], [1.0], color="red", s=80, zorder=5,
           label="max entropy at p=0.5")
ax.set_xlabel("p"); ax.set_ylabel("H(p) [bits]")
ax.set_title("Binary entropy H_b(p)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()


**最大熵 = 均匀分布**：
$$X \in \{1, \dots, K\} \text{ 均匀} \implies H(X) = \log_2 K$$

直观：均匀分布最"不确定"，所以传达最多信息。
Uniform = maximum uncertainty → maximum entropy.

### 一些熟例 / Some familiar values

| 分布 | $H(X)$ |
|---|---|
| 公平硬币 / fair coin | 1 bit |
| 公平骰子 / fair die | $\log_2 6 \approx 2.585$ bits |
| ASCII 一个字符 / one ASCII char | 8 bits (max) |
| 英语字母实际 / actual English letter | ~4.1 bits (much redundancy!) |


In [ ]:
# 验证：均匀分布熵最大 / Uniform max-entropy check
def entropy(p):
    p = np.asarray(p)
    p = p[p > 0]   # 0·log 0 := 0
    return float(-np.sum(p * np.log2(p)))

print(f"H(fair coin)  = {entropy([0.5, 0.5]):.4f}    (1 bit)")
print(f"H(fair die)   = {entropy([1/6]*6):.4f}    (≈ 2.585)")
print(f"H(biased coin p=0.1) = {entropy([0.1, 0.9]):.4f}")
print(f"H(very skewed p=[0.99, 0.01]) = {entropy([0.99, 0.01]):.4f}")


<a id="3"></a>
## 3. 联合熵 & 条件熵 / Joint & Conditional Entropy

### 3.1 联合熵 / Joint entropy

$$H(X, Y) = -\sum_{x,y} p(x, y) \log p(x, y)$$

### 3.2 条件熵 / Conditional entropy

"已知 $Y$ 后，$X$ 还剩多少不确定性":
$$H(X \mid Y) = -\sum_{x,y} p(x, y) \log p(x \mid y) = \mathbb{E}_Y\bigl[H(X \mid Y=y)\bigr]$$

### 3.3 链式法则 / Chain rule

$$H(X, Y) = H(Y) + H(X \mid Y) = H(X) + H(Y \mid X)$$

**几何上**：
$$H(X, Y) \;=\; H(X) + H(Y) - I(X; Y)$$
其中 $I$ 是互信息。
$H(X, Y)$ + $H(X)$ + $H(Y)$ 满足"集合并/交"般的关系：


<a id="4"></a>
## 4. 互信息 ⭐ / Mutual Information

$X$ 和 $Y$ 共享多少信息：
How much info $X$ and $Y$ share:

$$
\boxed{\;I(X; Y) \;=\; \sum_{x, y} p(x, y) \log \dfrac{p(x, y)}{p(x)\,p(y)} \;=\; D_{\mathrm{KL}}\bigl(p(x, y) \,\big\|\, p(x)p(y)\bigr)\;}
$$

等价形式 / Equivalent forms:
$$I(X; Y) = H(X) - H(X \mid Y) = H(Y) - H(Y \mid X) = H(X) + H(Y) - H(X, Y)$$

### 关键性质 / Key properties

- $I(X; Y) \ge 0$，等号 ⇔ $X \perp\!\!\!\perp Y$
- 对称 / Symmetric: $I(X; Y) = I(Y; X)$
- $I(X; X) = H(X)$

### 互信息 vs 相关系数 / MI vs Correlation

| 维度 / Aspect | 相关系数 $\rho$ | 互信息 $I$ |
|---|---|---|
| 仅捕获 / Captures | **线性关系** | **任意依赖** |
| 取值 / Range | $[-1, 1]$ | $[0, +\infty)$ |
| 独立 ⇒ 0? | ✅ | ✅ |
| 0 ⇒ 独立? | ❌（除非高斯）| ✅ |

> 💡 **特征选择面试题答案**：互信息 > 相关系数，因为它**抓得到非线性关系**。sklearn 有 `mutual_info_classif` 直接用。
> MI catches non-linear feature-target dependencies — preferred for feature selection.

### 韦恩图直觉 / Venn diagram intuition

```
   ┌──────────────────────────┐
   │    H(X, Y)              │
   │  ┌──────────┬──────────┐ │
   │  │  H(X|Y)  │  I(X;Y)  │ │   ← H(X) = H(X|Y) + I(X;Y)
   │  │  unique  │  shared  │ │
   │  │  to X    │  info    │ │
   │  ├──────────┴──────────┤ │
   │  │           H(Y|X)    │ │   ← unique to Y
   │  └─────────────────────┘ │
   └──────────────────────────┘
```


In [ ]:
# 数值演示 / Numerical demo
# X 是 0..3 均匀；Y = X 但 30% 概率被随机覆盖 / Y = X with 30% noise
n = 100_000
X = rng.integers(0, 4, size=n)
flip = rng.random(n) < 0.3
Y = np.where(flip, rng.integers(0, 4, size=n), X)

def empirical_dist(arr):
    vals, counts = np.unique(arr, return_counts=True)
    return counts / counts.sum()

def joint_dist(a, b):
    grid = np.zeros((4, 4))
    for x, y in zip(a, b):
        grid[x, y] += 1
    return grid / grid.sum()

H_X = entropy(empirical_dist(X))
H_Y = entropy(empirical_dist(Y))
J = joint_dist(X, Y)
H_XY = entropy(J.flatten())

I_XY = H_X + H_Y - H_XY
print(f"H(X)   = {H_X:.4f}")
print(f"H(Y)   = {H_Y:.4f}")
print(f"H(X,Y) = {H_XY:.4f}")
print(f"I(X;Y) = {I_XY:.4f}    (should be < H(X) and > 0)")
print(f"H(X|Y) = H(X) - I(X;Y) = {H_X - I_XY:.4f}")


<a id="5"></a>
## 5. KL 散度 ⭐ / Kullback–Leibler Divergence

**衡量两个分布的差异**。
**Measures how different two distributions are.**

$$
\boxed{\;D_{\mathrm{KL}}(p \,\|\, q) \;=\; \sum_x p(x) \log \dfrac{p(x)}{q(x)} \;=\; \mathbb{E}_{p}\!\left[\log\dfrac{p(X)}{q(X)}\right]\;}
$$

### 关键性质 / Key properties

1. **非负** / Non-negative: $D_{\mathrm{KL}}(p \,\|\, q) \ge 0$（Gibbs 不等式）
2. **等号** ⇔ $p = q$
3. **不对称** ⭐: $D_{\mathrm{KL}}(p \,\|\, q) \ne D_{\mathrm{KL}}(q \,\|\, p)$ —— 所以**不是距离**
4. **不满足三角不等式** —— 真的不是距离

### 编码角度的解读 / Coding interpretation

$D_{\mathrm{KL}}(p \,\|\, q)$ = "**如果用基于 $q$ 设计的编码去编码服从 $p$ 的数据，每个符号平均多用多少 bit**"。
The extra bits per symbol when encoding $p$-distributed data with a code optimized for $q$.

### 不对称的直觉：Forward vs Reverse KL

| 方向 / Direction | 名字 / Name | $p$ 是 / $p$ is | 行为 / Behavior |
|---|---|---|---|
| $D_{\mathrm{KL}}(p \,\|\, q)$ | **Forward / Inclusive** | data 真实 / data truth | $q$ "**覆盖** $p$ 的支撑集" mode-covering |
| $D_{\mathrm{KL}}(q \,\|\, p)$ | **Reverse / Exclusive** | data 真实 / data truth | $q$ "**集中到 $p$ 的一个 mode**" mode-seeking |

> 💡 **变分推断 / VAE 用 forward**；**KL-regularized RL / GAN 训练里有些用 reverse**。
> VAE uses forward KL; some GAN / KL-RL papers use reverse.


In [ ]:
# 演示 KL 的不对称 / Demonstrate KL asymmetry
def kl(p, q, eps=1e-12):
    p = np.asarray(p); q = np.asarray(q)
    return float(np.sum(p * np.log2((p + eps) / (q + eps))))

p = np.array([0.7, 0.2, 0.05, 0.05])
q = np.array([0.25, 0.25, 0.25, 0.25])

print(f"D_KL(p || q) = {kl(p, q):.4f} bits")
print(f"D_KL(q || p) = {kl(q, p):.4f} bits")
print("→ 显然不对称！/ clearly asymmetric")
print(f"\nIdentical distributions:")
print(f"D_KL(p || p) = {kl(p, p):.4f} bits   (should be 0)")


In [ ]:
# 可视化 forward vs reverse KL: 拟合 mixture-of-Gaussians 用单 Gaussian
# Forward (zero-avoiding, covers all modes) vs reverse (zero-forcing, picks one)
x = np.linspace(-5, 8, 1000)

def gauss(x, mu, sig):
    return np.exp(-0.5*((x-mu)/sig)**2) / (sig*np.sqrt(2*np.pi))

# 真分布 p: 两个高斯的混合 / True p: mixture of two Gaussians
p_dens = 0.5 * gauss(x, 0, 0.8) + 0.5 * gauss(x, 5, 0.8)

# Forward: 选 q 最小化 D_KL(p || q) → q 尽量覆盖 p 的支撑集
# Reverse: 选 q 最小化 D_KL(q || p) → q 集中到 p 的一个 mode

# 用网格搜索演示 / Grid search to find best mu, sigma
def kl_continuous(p, q, x):
    p = np.clip(p, 1e-12, None); q = np.clip(q, 1e-12, None)
    dx = x[1] - x[0]
    return np.sum(p * np.log2(p / q)) * dx

best_forward = (None, None, np.inf)
best_reverse = (None, None, np.inf)
for mu in np.linspace(-3, 7, 80):
    for sig in np.linspace(0.5, 5.0, 80):
        q_dens = gauss(x, mu, sig)
        f = kl_continuous(p_dens, q_dens, x)
        r = kl_continuous(q_dens, p_dens, x)
        if f < best_forward[2]: best_forward = (mu, sig, f)
        if r < best_reverse[2]: best_reverse = (mu, sig, r)

print(f"forward-KL minimizer: μ={best_forward[0]:.2f}, σ={best_forward[1]:.2f}")
print(f"reverse-KL minimizer: μ={best_reverse[0]:.2f}, σ={best_reverse[1]:.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x, p_dens, "k-", lw=2.5, label="true p (mixture)")
ax.plot(x, gauss(x, *best_forward[:2]), "b--", lw=2,
        label=f"min D_KL(p||q): cover both modes")
ax.plot(x, gauss(x, *best_reverse[:2]), "r--", lw=2,
        label=f"min D_KL(q||p): pick one mode")
ax.legend()
ax.set_title("Forward (mode-covering) vs Reverse (mode-seeking) KL")
ax.set_xlabel("x"); ax.set_ylabel("density")
ax.grid(alpha=0.3)
plt.show()


**清晰对比 / Clear contrast**:
- **Forward KL (p || q)**：$q$ 必须**覆盖** $p$ 所有支撑（penalty 在 $p > 0, q = 0$）→ 拟合出一个**宽**高斯横跨两个峰
- **Reverse KL (q || p)**：$q$ 倾向**集中到 $p$ 的高密度区**（penalty 在 $q > 0, p = 0$）→ 选一个 mode，**忽略**另一个

This is why **forward KL** in VAEs makes the learned posterior "blurry" (covers all modes), while **reverse KL** in some methods makes mode collapse.


<a id="6"></a>
## 6. 交叉熵 = 熵 + KL ⭐

**交叉熵 / Cross-entropy**:
$$H(p, q) = -\sum_x p(x) \log q(x) = \mathbb{E}_p[-\log q(X)]$$

**直觉**：用 $q$ 编码 $p$-数据时的**平均编码长度**。
**Intuition**: average code length when encoding $p$-distributed data with code based on $q$.

### 关键分解 / Key decomposition

$$
\boxed{\;H(p, q) \;=\; H(p) + D_{\mathrm{KL}}(p \,\|\, q)\;}
$$

**证明** / Proof:
$$H(p, q) = -\sum p \log q = -\sum p \log p + \sum p \log \tfrac{p}{q} = H(p) + D_{\mathrm{KL}}(p\|q)$$

### 这告诉我们什么 / What this tells us

最小化 $H(p, q)$（关于 $q$ 的参数）= 最小化 $D_{\mathrm{KL}}(p\|q)$（因为 $H(p)$ 是常数）。

So **minimizing cross-entropy ≡ minimizing forward KL**.

ML 里：$p$ = 真实标签分布（one-hot），$q$ = 模型输出 softmax。**这就是分类损失的全部数学**。
In ML: $p$ = one-hot ground truth, $q$ = model's softmax. **That's the entire math of classification loss.**


In [ ]:
# 数值验证 H(p,q) = H(p) + D_KL(p||q)
p = np.array([0.7, 0.2, 0.05, 0.05])
q = np.array([0.4, 0.3, 0.2, 0.1])

def cross_entropy(p, q, eps=1e-12):
    p = np.asarray(p); q = np.asarray(q)
    return float(-np.sum(p * np.log2(q + eps)))

ce = cross_entropy(p, q)
h_p = entropy(p)
kl_pq = kl(p, q)

print(f"H(p, q)             = {ce:.4f}")
print(f"H(p) + D_KL(p || q) = {h_p:.4f} + {kl_pq:.4f} = {h_p + kl_pq:.4f}")
print(f"equal? {np.isclose(ce, h_p + kl_pq)}")


<a id="7"></a>
## 7. 为什么分类用交叉熵不用 MSE / Why CE not MSE

**面试 ★★★★★ 高频题**。
**Very common interview question.**

### 答案 1：概率上有意义 / Probabilistic justification

对**伯努利输出**，**MLE 的负对数似然就是交叉熵**：
For Bernoulli output, MLE's negative log-likelihood **is** binary cross-entropy:

$$
-\log p(y \mid \mathbf{x}, \mathbf{w}) = -y\log \hat{y} - (1-y)\log(1-\hat{y})
$$

MSE 对应"高斯噪声"假设，对类别数据不合理。
MSE corresponds to Gaussian-noise assumption — wrong for categorical data.

### 答案 2：梯度更"健康" / Healthier gradients

**MSE + sigmoid** 在饱和区梯度 $\sigma'(z) = \sigma(1-\sigma)$ **消失**：
MSE + sigmoid: gradients **vanish** in saturated regions.

$$\frac{\partial}{\partial z}\bigl[\tfrac{1}{2}(\sigma(z) - y)^2\bigr] = (\sigma(z) - y)\,\underbrace{\sigma(z)(1-\sigma(z))}_{\to 0 \text{ when saturated}}$$

**CE + sigmoid**:
$$\frac{\partial}{\partial z}\bigl[-y\log\sigma(z) - (1-y)\log(1-\sigma(z))\bigr] = \sigma(z) - y$$

即"预测 - 真值"——**永远不会消失**！这就是为什么用 CE。
**"Prediction minus truth" — never vanishes!** That's why CE is preferred.

### 答案 3：凸 vs 非凸 / Convex vs non-convex

对线性模型 + sigmoid + MSE → **非凸**（多个局部最小）。
对线性模型 + sigmoid + CE → **凸**（保证全局最优）。


In [ ]:
# 比较两种损失的梯度强度 / Compare gradient magnitudes
def sigmoid(z): return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 200)
y_true = 1  # 标签是 1

# MSE: ∂/∂z [(σ(z) - 1)²/2] = (σ - 1)σ(1-σ)
mse_grad = (sigmoid(z) - 1) * sigmoid(z) * (1 - sigmoid(z))
# CE:  ∂/∂z [-log σ(z)] = σ(z) - 1
ce_grad = sigmoid(z) - 1

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(z, np.abs(mse_grad), "r-", lw=2, label="|MSE gradient|")
ax.plot(z, np.abs(ce_grad), "b-", lw=2, label="|CE gradient|")
ax.set_xlabel("pre-activation z (truth = 1)")
ax.set_ylabel("|gradient|")
ax.set_title("MSE gradient vanishes when z is very negative;\nCE keeps strong signal")
ax.legend()
ax.grid(alpha=0.3)
ax.axvspan(-6, -3, alpha=0.1, color="red", label="saturation zone")
plt.show()


**在 $z \to -\infty$**（模型自信地错）：
- MSE 梯度 → **0** ❌（学不到东西）
- CE 梯度 → **-1** ✅（强力修正）

这就是为什么 NN 分类层永远配 CE。
This is why NN classification heads always use CE.


<a id="8"></a>
## 8. Jensen 不等式 & Gibbs 不等式

### Jensen 不等式 / Jensen's inequality

$f$ **凸** ⇒ $\mathbb{E}[f(X)] \ge f(\mathbb{E}[X])$
$f$ **凹** ⇒ $\mathbb{E}[f(X)] \le f(\mathbb{E}[X])$

### Gibbs 不等式 / Gibbs inequality

$$D_{\mathrm{KL}}(p \,\|\, q) \ge 0, \quad \text{equality iff } p = q$$

**简证**（基于 Jensen + $\log$ 凹）：
$$
D_{\mathrm{KL}}(p\|q) = -\sum p \log(q/p) = \mathbb{E}_p\bigl[-\log(q/p)\bigr] \ge -\log \mathbb{E}_p\bigl[q/p\bigr] = -\log \sum q = -\log 1 = 0
$$

### 这是大半个信息论的基石

KL ≥ 0 推出 / Implies:
- 熵的连接（$H(X|Y) \le H(X)$，条件减熵）
- 互信息 ≥ 0
- 最大熵原理（uniform 最大熵）


<a id="9"></a>
## 9. 连续随机变量的熵 / Differential Entropy

连续 $X$ 的熵：
$$h(X) = -\int p(x) \log p(x)\,dx$$

⚠ **注意**：
- 离散熵恒 ≥ 0；**微分熵可以为负**（如 $\mathcal{N}(0, 0.1)$）
- 不像离散熵那样有"编码长度"语义，但 KL / 互信息仍然合法
- Discrete entropy is always ≥ 0; **differential entropy can be negative** (e.g., $\mathcal{N}(0, 0.1)$).

### 几个常用的微分熵 / Common differential entropies (in nats)

| 分布 / Distribution | $h(X)$ |
|---|---|
| Uniform $\mathcal{U}(a, b)$ | $\ln(b - a)$ |
| Normal $\mathcal{N}(\mu, \sigma^2)$ | $\tfrac{1}{2}\ln(2\pi e\,\sigma^2)$ ⭐ |
| Exponential $\mathrm{Exp}(\lambda)$ | $1 - \ln \lambda$ |

### 最大熵原理 / Maximum entropy principle

**给定 $\mathbb{E}[X] = \mu$ 和 $\mathrm{Var}(X) = \sigma^2$，使熵最大的连续分布是 $\mathcal{N}(\mu, \sigma^2)$**。
**Among all distributions with fixed mean and variance, the Gaussian has maximum entropy.**

这就是"高斯无处不在"的另一个理由——它是"最少假设" / minimum-assumption 的分布。
This is yet another reason Gaussians are everywhere: they make the fewest extra assumptions.


<a id="10"></a>
## 10. 信息论在 ML 里的应用 / ML Applications

| 应用 / Application | 信息论概念 / Concept |
|---|---|
| **分类损失 cross-entropy** ⭐ | $H(p_{\text{true}}, q_{\text{model}})$ |
| **决策树分裂** ⭐ | 信息增益 $= H(Y) - H(Y \mid X_j)$ = $I(X_j; Y)$ |
| **特征选择 mutual_info_classif** | 互信息 $I(X_j; Y)$ |
| **变分自编码器 VAE** | ELBO = $\mathbb{E}\log p(x\mid z) - D_{\mathrm{KL}}(q(z\mid x)\|p(z))$ |
| **GAN (JS-divergence)** | 对称化的 KL |
| **知识蒸馏 KD** | $D_{\mathrm{KL}}(p_{\text{teacher}} \| p_{\text{student}})$ |
| **RLHF（DPO/PPO 中的 KL 惩罚）** | KL 防止 policy 偏离参考太远 |
| **LLM 评估 perplexity** ⭐ | $\mathrm{PPL} = 2^{H} = e^{\text{NLL}}$ |
| **对比学习 InfoNCE** | $I(z_x; z_y)$ 的下界 |
| **t-SNE** | 最小化 KL 散度 |

### LLM Perplexity 详解 / LLM perplexity

对长度 $n$ 的文本 $\mathbf{w} = (w_1, \dots, w_n)$，模型预测每个 token 的概率 $p(w_i \mid w_{<i})$：

$$\mathrm{NLL}(\mathbf{w}) = -\frac{1}{n}\sum_i \log p(w_i \mid w_{<i})$$

$$\mathrm{PPL}(\mathbf{w}) = \exp(\mathrm{NLL}) = 2^H$$

**直觉**：PPL = "模型平均在多少个 token 之间犹豫"。
**Intuition**: PPL = "average number of tokens the model is choosing among."

- PPL = 1 → 完美预测（每次都 100% 对）
- PPL = $|V|$（词汇表大小）→ 随机猜
- GPT-2 在 wikitext-103 上 PPL ≈ 18；GPT-4 在难数据集上常 < 5

> 💡 **面试一句话**：perplexity 就是 entropy 的指数，衡量模型对下一个 token 的"困惑度"。
> Perplexity is just the exponential of entropy — how "confused" the model is about the next token.


In [ ]:
# 一个 LLM 的小演示：3 个候选 token，模型预测 + 真实选择
# A toy LLM scenario: 3 candidates, model probs vs truth
import math

# 假设词汇表只有 3 个 token / Tiny vocab of 3
# 真实序列下一个 token 的 one-hot 概率
y_true = [
    [1, 0, 0],   # token 0
    [0, 1, 0],   # token 1
    [0, 0, 1],   # token 2
    [1, 0, 0],   # token 0
]
# 模型预测概率 / Model predictions
q_pred = [
    [0.8, 0.1, 0.1],
    [0.2, 0.7, 0.1],
    [0.1, 0.2, 0.7],
    [0.5, 0.3, 0.2],
]

# 每步 NLL = -log q(y_true)
nlls = []
for true, q in zip(y_true, q_pred):
    correct_idx = true.index(1)
    nlls.append(-math.log(q[correct_idx]))   # natural log

mean_nll = sum(nlls) / len(nlls)
ppl = math.exp(mean_nll)

print(f"per-token NLL: {[round(x, 4) for x in nlls]}")
print(f"mean NLL     : {mean_nll:.4f}")
print(f"perplexity   : {ppl:.4f}    (random would be 3.0 for vocab=3)")


<a id="11"></a>
## 11. 实战：用信息增益在 Iris 上构建决策树 / Hands-on

把熵 + 条件熵 + 互信息 三件套用到决策树里。
Use entropy + conditional entropy + MI on a decision tree.

**决策树的分裂规则**：选择**信息增益**最大的 (feature, threshold)。
**Splitting rule**: pick the (feature, threshold) with largest **information gain**:

$$
\mathrm{IG}(X_j, t) = H(Y) - \bigl[\tfrac{n_{\text{left}}}{n} H(Y_{\text{left}}) + \tfrac{n_{\text{right}}}{n} H(Y_{\text{right}})\bigr]
$$

注意：$\mathrm{IG} = I(\mathbf{1}_{X_j \le t}; Y)$ — 互信息！

> **Iris 数据集**回顾：150 朵鸢尾花，4 特征（萼/瓣长宽），3 类。
> Iris recap: 150 flowers, 4 features (sepal/petal length/width), 3 classes.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target
feat_names = iris.feature_names
class_names = iris.target_names

def class_entropy(labels):
    if len(labels) == 0: return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log2(p)))

H_Y = class_entropy(y)
print(f"H(Y) before any split: {H_Y:.4f} bits   (log2(3) ≈ 1.585)")


In [ ]:
# 对每个特征找最佳分裂阈值 / For each feature find best threshold
def best_split(X, y):
    n, d = X.shape
    best = (-1, -1.0, -np.inf)   # (feature, threshold, info_gain)
    H_total = class_entropy(y)
    for j in range(d):
        # 候选阈值 = 排序后的中点 / candidate thresholds = midpoints
        vals = np.sort(np.unique(X[:, j]))
        candidates = (vals[:-1] + vals[1:]) / 2
        for t in candidates:
            left  = y[X[:, j] <= t]
            right = y[X[:, j] >  t]
            if len(left) == 0 or len(right) == 0:
                continue
            H_cond = (len(left)/n)*class_entropy(left) + (len(right)/n)*class_entropy(right)
            ig = H_total - H_cond
            if ig > best[2]:
                best = (j, t, ig)
    return best

j_star, t_star, ig_star = best_split(X, y)
print(f"best split: feature='{feat_names[j_star]}'  threshold={t_star:.3f}")
print(f"info gain : {ig_star:.4f} bits   (of total H(Y) = {H_Y:.4f})")


In [ ]:
# 看看分裂后两侧的类别分布 / Look at class distributions on each side
mask = X[:, j_star] <= t_star
print(f"Split: '{feat_names[j_star]}' <= {t_star:.3f}\n")

print("LEFT side:")
left_classes, left_counts = np.unique(y[mask], return_counts=True)
for c, k in zip(left_classes, left_counts):
    print(f"  {class_names[c]:<12} {k:3d}")
print(f"  entropy = {class_entropy(y[mask]):.4f} bits  → {'PURE!' if class_entropy(y[mask]) == 0 else ''}")

print("\nRIGHT side:")
right_classes, right_counts = np.unique(y[~mask], return_counts=True)
for c, k in zip(right_classes, right_counts):
    print(f"  {class_names[c]:<12} {k:3d}")
print(f"  entropy = {class_entropy(y[~mask]):.4f} bits")


**观察 / Observation**：
- **最佳第一次分裂 = `petal length ≤ ~2.45`** → **左侧全是 setosa**（熵 = 0，完美纯！）
- 这跟我们在 0.1 和 0.7 节看到的"花瓣维度可分性最强"的 EDA 结论一致
- 右侧只剩 versicolor + virginica（需要再分一次）

**信息增益**告诉决策树：在第一刀就直接切出 setosa，剩下两类再处理。
Info gain says: slice setosa off immediately, then sort the rest.


In [ ]:
# 跟 sklearn 默认决策树根节点对比 / Cross-check with sklearn
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=0).fit(X, y)
root = clf.tree_
sk_feat, sk_thresh = root.feature[0], root.threshold[0]
print(f"sklearn root: '{feat_names[sk_feat]}' <= {sk_thresh:.4f}")
print(f"our    root: '{feat_names[j_star]}' <= {t_star:.4f}")

# 二者其实都"完美分离 setosa" → 同分；只是 tie-breaking 顺序不同。
# Both perfectly split off setosa → tied; just different tie-breaking.
# 验证两个分裂的信息增益相等 / Confirm both splits have the same info gain
def ig_of(j, t):
    mask = X[:, j] <= t
    left, right = y[mask], y[~mask]
    if len(left) == 0 or len(right) == 0: return -np.inf
    return class_entropy(y) - (len(left)/len(y))*class_entropy(left) - (len(right)/len(y))*class_entropy(right)

print(f"\nIG of sklearn split: {ig_of(sk_feat, sk_thresh):.6f}")
print(f"IG of our    split: {ig_of(j_star, t_star):.6f}")
print(f"→ tied? {np.isclose(ig_of(sk_feat, sk_thresh), ig_of(j_star, t_star))}")


**两个分裂的信息增益完全相同**——它们都"一刀切出 setosa"，是**并列最优**。sklearn 用某种 tie-breaking 顺序（按特征/阈值排序）选了另一个等价方案。
**Two splits tie at the same info gain** — both perfectly carve setosa off. sklearn picks the other equally optimal split via its tie-breaking order.

这就是手撕信息增益的价值：你完全明白决策树底下在做什么，遇到 sklearn"奇怪"的选择也能解释。
This is the value of hand-computing info gain — you can explain "weird" sklearn picks because you know the criterion.


<a id="12"></a>
## 12. 小结 / Summary

### 概念地图 / Concept map

```
自信息 I(x) = -log p(x)
  │
  └── 期望 → 熵 H(X) = E[-log p(X)]
            ├── 二元熵函数 H_b(p)：在 p=0.5 时最大
            ├── 联合熵 H(X, Y)
            ├── 条件熵 H(X | Y) → "知道 Y 后还剩多少不确定性"
            ├── 互信息 I(X;Y) = H(X) - H(X|Y) = D_KL(p(x,y)||p(x)p(y))
            │     └── 推决策树 / 特征选择
            │
            └── 跨分布
                  ├── KL 散度 D_KL(p||q)：不对称、≥0 (Gibbs)
                  │     ├── forward KL → mode covering
                  │     └── reverse KL → mode seeking
                  │
                  └── 交叉熵 H(p,q) = H(p) + D_KL(p||q) ⭐
                        ├── 分类 NN 损失 = CE
                        └── LLM perplexity = exp(NLL) = 2^H
```

### 🧠 必记公式 / Must-know formulas

| 量 | 公式 |
|---|---|
| 自信息 | $I(x) = -\log p(x)$ |
| 熵 | $H(X) = -\sum p \log p$ |
| 条件熵 | $H(X\mid Y) = -\sum_{x,y} p(x,y)\log p(x\mid y)$ |
| 链式 | $H(X, Y) = H(Y) + H(X\mid Y)$ |
| 互信息 | $I(X;Y) = H(X) - H(X\mid Y) = D_{\mathrm{KL}}(p(x,y)\|p(x)p(y))$ |
| KL | $D_{\mathrm{KL}}(p\|q) = \sum p \log(p/q) \ge 0$ |
| 交叉熵 | $H(p,q) = H(p) + D_{\mathrm{KL}}(p\|q)$ ⭐ |
| Perplexity | $\mathrm{PPL} = 2^H = e^{\mathrm{NLL}}$ |
| 高斯微分熵 | $h = \tfrac{1}{2}\ln(2\pi e\,\sigma^2)$ |

### 💡 工业速查 / Industry cheat sheet

```python
# 分类损失（NN）/ NN classification loss
import torch.nn.functional as F
loss = F.cross_entropy(logits, target)        # combines softmax + NLL

# 二分类 / Binary
loss = F.binary_cross_entropy_with_logits(z, target)

# KL（已是 log-probs）/ KL on log-probs
loss = F.kl_div(log_q, p, reduction='batchmean')

# 互信息特征选择 / MI feature selection
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X, y)

# 决策树用 entropy / Tree with entropy criterion
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion='entropy')

# 计算 perplexity / Compute perplexity
import math
ppl = math.exp(mean_nll_per_token)
```

### 💡 面试速查 / Interview must-knows

1. **熵 = 期望不确定性 / 编码下界**（两种解释二选一就够答）
2. **CE = H + KL**：分类用 CE 因为 (a) MLE 视角 (b) sigmoid 梯度不消失 (c) 凸性
3. **KL 不对称** ⇒ forward = mode-covering, reverse = mode-seeking（VAE 用 forward）
4. **互信息 vs 相关系数**：MI 抓任意依赖；ρ 只抓线性
5. **PPL = 2^H**：一句话答 LLM 评估
6. **决策树信息增益 = 互信息**

### 下一节预告 / Next up

**Part 0.12 · 工程化** —— Git / venv / Jupyter / VSCode 的 DS 实战流程，把 Part 0 全部串到一起做完一个 ready-to-deploy 的项目模板。
**Part 0.12 · Engineering Basics** — Git / venv / Jupyter / VSCode in a DS workflow. Wraps up Part 0 with a ready-to-deploy project template.
